## With FPGA binarization

In [ ]:
from pynq import Overlay, allocate
from pynq.lib.video import VideoMode
import cv2
import numpy as np
import time

print("Loading the binarization hardware...")
overlay_custom = Overlay("design_1_wrapper.bit") 
dma = overlay_custom.axi_dma_0

# Citire imagine grayscale
img_path = "images.bmp"
gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

# Verificam daca citirea a reusit
if gray is None:
    raise ValueError("Imaginea couldn't be found / read!")

# resize to 1280x720
gray = cv2.resize(gray, (1280, 720))


height, width = gray.shape
in_buf = allocate(shape=(height, width), dtype=np.uint8)
out_buf = allocate(shape=(height, width), dtype=np.uint8)

# we copy the data to the input buffer
in_buf[:] = gray
start = time.time()

print("Starting DMA processing...")
dma.sendchannel.transfer(in_buf)
dma.recvchannel.transfer(out_buf)
dma.sendchannel.wait()
dma.recvchannel.wait()

end = time.time()

print(f"Processing complete. Time taken: {end - start:.5f} seconds")

# we save the result in RAM
result_safe = out_buf.copy()

# we clean old hanging DMA memory
in_buf.freebuffer()
out_buf.freebuffer()


base = Overlay("base.bit")

hdmi_out = base.video.hdmi_out
mode = VideoMode(1280, 720, 24)
hdmi_out.configure(mode)
hdmi_out.start()


final_frame = np.stack((result_safe,)*3, axis=-1)

# Afisare
print("Displaying...")
outframe = hdmi_out.newframe()
outframe[:] = final_frame
hdmi_out.writeframe(outframe)

## Without FPGA binarization

In [ ]:
from pynq import Overlay, allocate
from pynq.lib.video import VideoMode
import cv2
import numpy as np
import time

base = Overlay("base.bit")

img_path = "images.bmp"
gray = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

if gray is None:
    raise ValueError("Error!")

gray = cv2.resize(gray, (1280, 720))

height, width = gray.shape


in_buf = allocate(shape=(height, width), dtype=np.uint8)
out_buf = allocate(shape=(height, width), dtype=np.uint8)

in_buf[:] = gray

print("Starting binarization")
start = time.time()

for y in range(height):
    for x in range(width):
        val = in_buf[y, x]
        

        if val > 128:
            out_buf[y, x] = 255
        else:
            out_buf[y, x] = 0


end = time.time()
print(f"Processing complete. Time taken: {end - start:.5f} seconds")

result_safe = out_buf.copy()

hdmi_out = base.video.hdmi_out
mode = VideoMode(1280, 720, 24) # 24 bits per pixel (RGB)
hdmi_out.configure(mode)
hdmi_out.start()


final_frame = np.stack((result_safe,)*3, axis=-1)


outframe = hdmi_out.newframe()
outframe[:] = final_frame
hdmi_out.writeframe(outframe)


print("Done.")